# **DeconvNet: провести отклик канала обратно к пикселям**

Практика к модулю [«Анализ CNN-based моделей»](https://ai-interpretability.school), урок «DeconvNet: что зажгло канал на этой картинке».

В уроке DeconvNet ([Zeiler и Fergus, 2013](https://arxiv.org/abs/1311.2901)) описан так: берут отклик одного канала в одной позиции, обнуляют все остальное и ведут оставшееся число вниз по сети, обращая каждую операцию — свертку транспонированной сверткой, пулинг через запомненные позиции максимумов, ReLU выпрямлением. Здесь мы соберем это руками, сверим с `captum.attr.Deconvolution` и проверим, чему такой картинке можно верить:

1. AlexNet и снимок, на котором будем смотреть;
2. DeconvNet руками: одно правило для ReLU на обратном проходе;
3. библиотечный вызов Captum и сверка;
4. три правила для ReLU рядом: градиент, DeconvNet, guided backpropagation;
5. разные слои и каналы;
6. где ломается: картинка почти не зависит от класса и от весов верхних слоев.

AlexNet — ближайшая к сети статьи Zeiler и Fergus архитектура из torchvision. Тетрадь идет на процессоре; весь прогон занимает около минуты, веса AlexNet (около 230 МБ) скачиваются при первом запуске.

Автор фото кошки и мопса: [Priscilla Du Preez, Unsplash](https://unsplash.com/photos/qtkpOxCpl0Y), лицензия Unsplash.

In [ ]:
!pip install -q captum==0.9.0

In [ ]:
import io
import copy
import urllib.request
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # tqdm без ipywidgets; в Colab ipywidgets есть
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from captum.attr import Deconvolution, GuidedBackprop, Saliency, NeuronDeconvolution

warnings.filterwarnings('ignore', category=UserWarning, module='captum')   # Captum предупреждает, что сам включает градиент по входу и ставит хуки
torch.manual_seed(0)

model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1).eval()
for m in model.modules():
    if isinstance(m, torch.nn.ReLU):
        m.inplace = False            # хукам на обратном проходе нужны неперезаписанные тензоры
print(model.features)

## 1. Снимок

Снимок с кошкой и собакой из папки `data` репозитория курса. Подготовка стандартная для ImageNet: сторона 256, центральный квадрат 224, нормировка по средним и разбросам каналов ImageNet.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
prep = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                           transforms.Normalize(MEAN, STD)])
raw = urllib.request.urlopen('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg', timeout=30).read()
x = prep(Image.open(io.BytesIO(raw)).convert('RGB')).unsqueeze(0)
labels = models.AlexNet_Weights.IMAGENET1K_V1.meta['categories']

with torch.no_grad():
    probs = model(x).softmax(1)[0]
top5 = probs.topk(5)
for p, i in zip(top5.values, top5.indices):
    print(f'{p:.3f}  {i.item():4d}  {labels[i]}')


def to_img(t):
    # тензор входа (1, 3, H, W) → картинка в [0, 1] для imshow
    return (t[0].permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)).clamp(0, 1).numpy()


def to_map(a):
    # атрибуция (1, 3, H, W) → картинка: каждый канал нормирован на свой наибольший модуль
    a = a[0].detach().permute(1, 2, 0).numpy()
    return np.clip(0.5 + a / (2 * np.abs(a).max() + 1e-12), 0, 1)


def corr(a, b):
    return np.corrcoef(a.detach().flatten().numpy(), b.detach().flatten().numpy())[0, 1]

## 2. DeconvNet руками

Как обратить каждую операцию:

- **свертка** обращается транспонированной сверткой с теми же фильтрами — ровно это делает обратный проход autograd через `Conv2d`;
- **max-пулинг** обращается через запомненные позиции максимумов (switches): значение возвращается туда, откуда пришел максимум, остальные позиции получают ноль — это тоже обычный обратный проход через `MaxPool2d`;
- **ReLU** — единственное место, где DeconvNet отличается от градиента. Градиент пропускает сигнал там, где **на прямом проходе** вход ReLU был положительным. DeconvNet смотрит не на прямой проход, а на сам обратный сигнал и пропускает его положительную часть: $R_{\text{in}}=\max(R_{\text{out}},0)$.

Значит, весь DeconvNet — это обычный обратный проход, в котором у каждого ReLU заменено правило. Заменим его хуком `register_full_backward_hook`: хук получает градиент по выходу модуля и возвращает то, что пойдет дальше вниз.

Цель — канал последнего сверточного блока `features[11]` (ReLU после пятой свертки), в позиции, где этот канал откликается сильнее всего. Выбираем канал с наибольшим откликом на снимке.

In [ ]:
LAYER = model.features[11]
captured = {}
hook = LAYER.register_forward_hook(lambda m, i, o: captured.__setitem__('a', o))
with torch.no_grad():
    model(x)
hook.remove()
A = captured['a'][0]                                     # (256, 13, 13)
CHANNEL = A.amax(dim=(1, 2)).argmax().item()
POS = tuple(int(p) for p in np.unravel_index(A[CHANNEL].argmax().item(), A[CHANNEL].shape))
print(f'канал {CHANNEL}, позиция {POS} в карте {tuple(A.shape[1:])}, отклик {A[CHANNEL][POS]:.2f}')

**Задание 1. Напишите функцию `backward_with_rule(rule, layer, channel, pos)`. Она ставит на каждый `nn.ReLU` модели `model` хук обратного прохода с правилом `rule` (функция от градиента по выходу и сохраненного входа ReLU), делает прямой проход до слоя `layer`, берет отклик `channel` в позиции `pos` (все остальное обнуляется само: мы дифференцируем одно число), запускает обратный проход до входа и возвращает градиент по входу, сняв хуки. Правило DeconvNet — `lambda g, inp: g.clamp(min=0)`. Посчитайте `deconv_hand` для канала `CHANNEL` в позиции `POS` и нарисуйте снимок рядом с картинкой.**

Подсказка: вход ReLU на прямом проходе можно запомнить forward-хуком того же модуля; для DeconvNet он не нужен, но пригодится в разделе 4.

In [ ]:
# Ваш код здесь

## 3. Библиотечный вызов

В Captum DeconvNet для выхода модели — `captum.attr.Deconvolution(model).attribute(x, target=класс)`, для внутреннего нейрона — `NeuronDeconvolution(model, layer).attribute(x, neuron_selector=(канал, строка, столбец))`. Captum ставит хуки на модули `nn.ReLU`, поэтому функциональные вызовы `F.relu` внутри `forward` он не увидит, а модуль ReLU, использованный дважды, получит правило дважды; у AlexNet каждый ReLU — отдельный модуль.

In [ ]:
deconv_captum = NeuronDeconvolution(model, LAYER).attribute(x, neuron_selector=(CHANNEL,) + POS)
print(f'корреляция ручного DeconvNet и NeuronDeconvolution: {corr(deconv_hand, deconv_captum):.4f}')
print(f'наибольшее расхождение: {(deconv_hand - deconv_captum).abs().max():.2e}')

## 4. Три правила для ReLU

Метод визуализации через обратный проход определяется тем, что делает ReLU с сигналом:

| правило | что пропускает ReLU | формула |
|---|---|---|
| градиент (vanilla gradient) | там, где вход ReLU на прямом проходе положителен | $g\cdot[z>0]$ |
| DeconvNet | положительную часть обратного сигнала | $\max(g,0)$ |
| guided backpropagation ([Springenberg и др., 2015](https://arxiv.org/abs/1412.6806)) | оба условия сразу | $\max(g,0)\cdot[z>0]$ |

**Задание 2. Посчитайте функцией `backward_with_rule` картинки градиента и guided backpropagation для того же канала и позиции и нарисуйте три картинки рядом. Выведите попарные корреляции трех картинок.**

In [ ]:
# Ваш код здесь

## 5. Разные слои

Zeiler и Fergus показывали, что чем выше слой, тем крупнее и сложнее структура, которую восстанавливает DeconvNet: в первых слоях — края и цветовые пятна, в последних — части объектов. Возьмем самый сильный канал на выходе каждого из пяти сверточных блоков (ReLU `features[1]`, `[4]`, `[7]`, `[9]`, `[11]`) и для каждого посчитаем DeconvNet от его максимальной позиции. Рамкой на снимке показана область, где картинка DeconvNet отлична от нуля, — фактическое рецептивное поле этого отклика.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7.5))
for k, idx in enumerate((1, 4, 7, 9, 11)):
    layer = model.features[idx]
    hook = layer.register_forward_hook(lambda m, i, o: captured.__setitem__('a', o))
    with torch.no_grad():
        model(x)
    hook.remove()
    Al = captured['a'][0]
    ch = Al.amax(dim=(1, 2)).argmax().item()
    pos = tuple(int(p) for p in np.unravel_index(Al[ch].argmax().item(), Al[ch].shape))
    a = backward_with_rule(lambda g, z: g.clamp(min=0), layer, ch, pos)
    nz = np.argwhere(a[0].abs().sum(0).numpy() > 0)
    (r0, c0), (r1, c1) = nz.min(0), nz.max(0)
    axes[0, k].imshow(to_img(x))
    axes[0, k].add_patch(plt.Rectangle((c0, r0), c1 - c0, r1 - r0, fill=False, color='#c0392b', lw=2))
    axes[0, k].set_title(f'features[{idx}], channel {ch}\nnonzero region {r1 - r0 + 1}×{c1 - c0 + 1} px')
    axes[1, k].imshow(to_map(a)[r0:r1 + 1, c0:c1 + 1])
    axes[1, k].set_title('DeconvNet, cropped to the region')
for ax in axes.flat:
    ax.axis('off')
fig.tight_layout()
plt.show()

## 6. Где ломается

Картинка DeconvNet красивая и «похожа на объект». Проверим, отвечает ли она на вопросы, которые мы ей задаем. Две проверки:

- **чувствительность к классу.** Посчитаем `Deconvolution` для самого вероятного класса (`pug`, собака) и для класса `tabby` (полосатая кошка) — на снимке есть и собака, и кошка. Если объяснение про класс, картинки для двух классов должны различаться;
- **чувствительность к весам** ([Adebayo и др., 2018](https://arxiv.org/abs/1810.03292)). Заменим веса классификатора и последней свертки случайными. Если объяснение про то, чему научилась сеть, картинка должна измениться.

Для сравнения те же проверки — у градиента (`Saliency`) и guided backpropagation (`GuidedBackprop`).

**Задание 3. Для каждого из трех методов (`Saliency` с `abs=False`, `Deconvolution`, `GuidedBackprop`) посчитайте корреляцию картинок для классов `c1 = top5.indices[0]` и `c2 = labels.index('tabby')` и корреляцию картинки для класса `top5.indices[0]` у исходной модели и у модели `random_model`, где веса `classifier` и `features[10]` заменены нормальным шумом со стандартным отклонением 0.01 (`torch.nn.init.normal_`, после `torch.manual_seed(0)`). Соберите таблицу.**

In [ ]:
# Ваш код здесь

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for row, (name, method) in enumerate((('Gradient', Saliency), ('DeconvNet', Deconvolution), ('Guided backprop', GuidedBackprop))):
    for col, (title, net, tgt) in enumerate(((f'class {labels[c1]}', model, c1), (f'class {labels[c2]}', model, c2),
                                             (f'class {labels[c1]}, random top layers', random_model, c1))):
        axes[row, col].imshow(to_map(attribution(method, net, tgt)))
        axes[row, col].set_title(f'{name}\n{title}', fontsize=9)
        axes[row, col].axis('off')
fig.tight_layout()
plt.show()

**Задание 4. По таблице задания 3: какой из трех методов прошел обе проверки и что это говорит о том, откуда берется картинка DeconvNet?**

**Ваш ответ:**

## Что унести из тетради

- **DeconvNet — это обратный проход с другим правилом для ReLU.** Свертку и пулинг обращает обычный autograd, а вся разница с градиентом — в том, что ReLU пропускает положительную часть обратного сигнала.
- **Ручная версия совпадает с `captum.attr.NeuronDeconvolution`.** Captum ставит те же хуки на модули `nn.ReLU`; функциональный `F.relu` он не увидит.
- **Чем выше слой, тем больше область снимка за откликом.** Это рецептивное поле отклика, и DeconvNet показывает, какие пиксели внутри него участвовали.
- **Картинка DeconvNet почти не зависит от класса и от весов верхних слоев.** Она хорошо показывает, на что в снимке смотрят нижние слои, и плохо — почему сеть приняла решение. Для вопроса о решении нужны методы, которые проходят проверку рандомизацией весов.

Источники:
1. [Zeiler & Fergus, 2013. Visualizing and understanding convolutional networks](https://arxiv.org/abs/1311.2901)
2. [Springenberg et al., 2015. Striving for simplicity: the all convolutional net](https://arxiv.org/abs/1412.6806)
3. [Adebayo et al., 2018. Sanity checks for saliency maps](https://arxiv.org/abs/1810.03292)
4. [Nie, Zhang & Patel, 2018. A theoretical explanation for perplexing behaviors of backpropagation-based visualizations](https://arxiv.org/abs/1805.07039)
5. [Captum: Deconvolution](https://captum.ai/api/deconvolution.html)